# Task 2.2 — Learning Rate Sweep on the Tiny Model

Train the Tiny model (1.05M params) at 7 learning rates on a log scale to identify the optimal LR.
Each trial runs for 20% of one epoch (~310 steps, ~4 min) to save compute while still being representative.

**LRs tested:** 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2

In [ ]:
import os, sys
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), 'ML-Final-Project'))
if not os.path.exists('configs'):
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scripts.train import train_model

LEARNING_RATES = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]

## Option A: Load Already-Completed Results
The sweep was already run via `scripts/lr_sweep.py`. Load those results directly.

In [ ]:
results_path = 'outputs/results/lr_sweep_results.json'

if os.path.exists(results_path):
    with open(results_path) as f:
        raw = json.load(f)
    # Convert string keys back to floats
    results = {float(k): v for k, v in raw.items()}
    print('Loaded existing LR sweep results:')
    print(f"  {'LR':>10}  {'Val Loss':>10}")
    print('  ' + '-'*24)
    for lr, val in sorted(results.items()):
        marker = '  ← BEST' if val == min(results.values()) else ''
        print(f"  {lr:>10.2e}  {val:>10.4f}{marker}")
    best_lr = min(results, key=results.get)
    print(f'\nBest LR: {best_lr:.2e}  (val_loss = {results[best_lr]:.4f})')
else:
    print('No results file found. Run Option B below to execute the sweep.')

## Option B: Re-run the Sweep
Skip this cell if results are already loaded above. Set `sweep_fraction` to control how much of an epoch to run per LR.

In [ ]:
# Uncomment to re-run the sweep
# sweep_fraction = 0.2  # 20% of epoch per LR (~4 min each, ~30 min total)
# full_steps = 1551
# max_steps = max(50, int(full_steps * sweep_fraction))

# results = {}
# for lr in LEARNING_RATES:
#     print(f'\n--- LR = {lr:.1e} ---')
#     r = train_model(
#         config_path='configs/tiny.json',
#         lr=lr,
#         data_dir='data/tokenized',
#         output_dir='outputs',
#         eval_interval=50,
#         max_steps=max_steps,
#         save_checkpoint=False,
#     )
#     results[lr] = r['val_loss']
#     print(f'  val_loss = {r["val_loss"]:.4f}')

# best_lr = min(results, key=results.get)
# print(f'\nBest LR: {best_lr:.2e}')
print('(Skipped — results already loaded from file)')

## Plot: LR vs. Validation Loss

In [ ]:
lrs  = sorted(results.keys())
vals = [results[lr] for lr in lrs]
best_lr  = min(results, key=results.get)
best_val = results[best_lr]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(lrs, vals, 'o-', color='steelblue', linewidth=2, markersize=8,
            label='Val loss')
ax.axvline(best_lr, color='tomato', linestyle='--', linewidth=1.8,
           label=f'Best LR = {best_lr:.1e}  (val = {best_val:.4f})')

# Annotate each point
for lr, v in zip(lrs, vals):
    ax.annotate(f'{v:.3f}', (lr, v), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8, color='navy')

ax.set_xlabel('Learning Rate (log scale)', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title('Task 2.2 — LR Sweep on Tiny Model (20% epoch per trial)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()

os.makedirs('outputs/plots', exist_ok=True)
fig.savefig('outputs/plots/lr_sweep.png', dpi=150)
plt.show()
print('Plot saved to outputs/plots/lr_sweep.png')

In [ ]:
# Show training curves for all 7 LR trials
import glob

log_files = sorted(glob.glob('outputs/logs/tiny_lr*.json'))
if log_files:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(log_files)))
    for path, color in zip(log_files, colors):
        with open(path) as f:
            log = json.load(f)
        lr  = log['lr']
        entries = log.get('train_losses', [])
        if not entries: continue
        steps  = [e['step']       for e in entries]
        t_loss = [e['train_loss'] for e in entries]
        v_loss = [e['val_loss']   for e in entries]
        label  = f'lr={lr:.1e}'
        axes[0].plot(steps, t_loss, label=label, color=color, linewidth=1.5)
        axes[1].plot(steps, v_loss, label=label, color=color, linewidth=1.5)

    for ax, title in zip(axes, ['Train Loss', 'Val Loss']):
        ax.set_xlabel('Step'); ax.set_ylabel('Loss')
        ax.set_title(f'Tiny — {title} vs Step (all LRs)', fontsize=11)
        ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig('outputs/plots/lr_sweep_curves.png', dpi=150)
    plt.show()
else:
    print('No log files found — run the training to generate curves.')

## Summary

| LR | Val Loss |
|---|---|
| 1e-5 | 6.0661 |
| 3e-5 | 5.4399 |
| 1e-4 | 4.2819 |
| 3e-4 | 3.0435 |
| 1e-3 | 2.6896 |
| **3e-3** | **2.3214** ← best |
| 1e-2 | 2.3567 |

The loss decreases monotonically from 1e-5 → 3e-3, then slightly increases at 1e-2, indicating 1e-2 is just over the stability threshold. **Best LR = 3e-3** will be used for all model sizes in Task 2.3.